# 07 — HMRNet Coherence Optimization

## Objective

This notebook optimizes **HMRNet-Full** to increase intrinsic topic coherence, particularly \(C_v\), **without sacrificing the main strengths already observed in the previous experiments**:

- high topic diversity,
- low topic redundancy,
- high topic stability,
- useful downstream topic representations.

The optimization is deliberately conservative: the core HMRNet architecture is kept unchanged. The notebook first tunes the loss weights that were implicated by the ablation study:

1. stability-loss weight,
2. alignment-loss weight,
3. diversity-loss weight,
4. KL-loss weight,

then optionally performs a small local joint search.

### Selection principle

A candidate is not selected from \(C_v\) alone. During screening, the notebook prefers high \(C_v\) subject to diversity/redundancy constraints. The strongest candidates are then re-evaluated across multiple random seeds. The preferred configuration is the candidate with the highest coherence that still satisfies a minimum topic-stability threshold.

> **Recommended workflow:** run once in `MODE="quick"` to verify the pipeline. Then switch to `MODE="final"` for the paper-quality experiment.

### Expected outputs

- `experiment_07_all_tuning_runs.csv`
- `experiment_07_stage_*.csv`
- `experiment_07_candidate_stability.csv`
- `experiment_07_final_metrics.csv`
- `experiment_07_best_config.json`
- `HMRNet-Optimized.pt`
- optimization figures and checkpoints

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Locate the HMRNet project

The cell below preserves the project location used in the previous notebooks whenever possible.

In [ ]:
from pathlib import Path
import sys

candidate_paths = [
    Path("/content/drive/MyDrive/UDB/Riset/2026"),

]

PROJECT_ROOT = None

for candidate in candidate_paths:
    if candidate.exists() and (candidate / "src/hmrnet.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "HMRNet project folder was not found. "
        "Update candidate_paths with the correct Google Drive location."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"
OPT_DIR = CHECKPOINTS_DIR / "coherence_optimization"
OPT_MODELS_DIR = OPT_DIR / "models"
OPT_RESULTS_DIR = OPT_DIR / "results"
OPT_STABILITY_DIR = OPT_DIR / "stability"

for directory in [
    TABLES_DIR,
    FIGURES_DIR,
    OPT_DIR,
    OPT_MODELS_DIR,
    OPT_RESULTS_DIR,
    OPT_STABILITY_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_DIR:", FEATURE_DIR)
print("OPT_DIR:", OPT_DIR)

## 3. Install/verify dependencies

In [ ]:
%cd {PROJECT_ROOT}

%pip install -q \
    "gensim==4.4.0" \
    openpyxl

In [ ]:
import gc
import copy
import inspect
import itertools
import json
import math
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from src.preprocessing import load_feature_bundle
from src.hmrnet import HMRNet, HMRNetConfig
from src.topic_metrics import evaluate_topics
from src.stability_metrics import evaluate_stability_runs

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Load the fixed feature bundle

The **same feature space** from Experiment 00 is reused so that tuning does not alter the vocabulary or representation pipeline.

In [ ]:
bundle = load_feature_bundle(FEATURE_DIR)

raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

n_documents = len(raw_texts)

assert bow.shape[0] == n_documents
assert semantic_embeddings.shape[0] == n_documents
assert document_embeddings.shape[0] == n_documents

print("Documents:", n_documents)
print("Vocabulary:", len(vocabulary))
print("BoW:", bow.shape)
print("Semantic embeddings:", semantic_embeddings.shape)
print("Document embeddings:", document_embeddings.shape)

## 5. Load the original HMRNet-Full configuration

Instead of hard-coding the original configuration, the notebook reads it from the trained `HMRNet-Full.pt` checkpoint. This reduces the risk of accidentally changing unrelated architecture settings.

In [ ]:
ORIGINAL_MODEL_PATH = CHECKPOINTS_DIR / "HMRNet-Full.pt"

if not ORIGINAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{ORIGINAL_MODEL_PATH} was not found. "
        "Complete Experiment 01 first."
    )

original_model = HMRNet.load(
    ORIGINAL_MODEL_PATH,
    device=DEVICE,
)

if not hasattr(original_model, "config"):
    raise AttributeError(
        "Loaded HMRNet model has no `config` attribute. "
        "Inspect src/hmrnet.py and adapt this cell."
    )

BASE_CONFIG = copy.deepcopy(original_model.config)

def config_to_dict(config):
    if hasattr(config, "__dict__"):
        return {
            key: value
            for key, value in vars(config).items()
            if not key.startswith("_")
        }
    return {}

base_config_dict = config_to_dict(BASE_CONFIG)

print("Original HMRNet configuration:")
for key in sorted(base_config_dict):
    print(f"  {key}: {base_config_dict[key]}")

del original_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 6. Resolve configuration-field names safely

The generated HMRNet source may use slightly different naming conventions. The helper below supports common aliases and reports the exact field found in your project.

In [ ]:
CONFIG_ALIASES = {
    "stability_weight": [
        "stability_weight", "lambda_stability", "lambda_sl", "sl_weight"
    ],
    "alignment_weight": [
        "alignment_weight", "lambda_alignment", "lambda_al", "al_weight"
    ],
    "diversity_weight": [
        "diversity_weight", "lambda_diversity", "lambda_dl", "dl_weight"
    ],
    "kl_weight": [
        "kl_weight", "lambda_kl", "beta_kl", "kl_beta"
    ],
    "epochs": [
        "epochs", "n_epochs", "max_epochs"
    ],
    "batch_size": [
        "batch_size"
    ],
    "random_state": [
        "random_state", "seed"
    ],
    "n_topics": [
        "n_topics", "num_topics", "n_components"
    ],
    "patience": [
        "patience", "early_stopping_patience"
    ],
    "device": [
        "device"
    ],
}

def resolve_config_attr(config, concept, required=True):
    available = set(config_to_dict(config))
    for candidate in CONFIG_ALIASES.get(concept, [concept]):
        if candidate in available or hasattr(config, candidate):
            return candidate

    if required:
        raise AttributeError(
            f"No configuration field found for '{concept}'. "
            f"Tried: {CONFIG_ALIASES.get(concept, [concept])}. "
            f"Available fields: {sorted(available)}"
        )
    return None

resolved_fields = {}

for concept in CONFIG_ALIASES:
    attr = resolve_config_attr(
        BASE_CONFIG,
        concept,
        required=concept in {
            "stability_weight",
            "alignment_weight",
            "diversity_weight",
            "kl_weight",
            "epochs",
            "random_state",
            "n_topics",
        },
    )
    resolved_fields[concept] = attr

print("Resolved fields:")
for concept, attr in resolved_fields.items():
    print(f"  {concept:20s} -> {attr}")

## 7. Experiment mode and optimization settings

`quick` is intended only for pipeline validation. `final` uses a broader search and should be used for the manuscript.

The default optimization keeps the original number of topics fixed. This preserves comparability with the main baseline experiment.

In [ ]:
MODE = "full"          # change to "final" for the paper-quality run
FORCE_RETRAIN = False   # True only when you intentionally want to overwrite cached runs

TOP_N_WORDS = 10
TUNING_SEED = 42
VALIDATION_SIZE = 0.20

MIN_TOPIC_DIVERSITY = 0.95
MAX_TOPIC_REDUNDANCY = 0.02
MIN_TOPIC_STABILITY = 0.95

RUN_LOCAL_JOINT_SEARCH = True
RUN_FINAL_10_SEED_STABILITY = (MODE == "final")

base_epochs_attr = resolved_fields["epochs"]
ORIGINAL_EPOCHS = int(getattr(BASE_CONFIG, base_epochs_attr))

if MODE == "quick":
    TUNING_EPOCHS = min(10, ORIGINAL_EPOCHS)
    CANDIDATE_STABILITY_SEEDS = [1, 2, 3]

    SL_CANDIDATES = [0.0, 0.005, 0.01, 0.02]
    AL_CANDIDATES = [0.0, 0.025, 0.05, 0.10]
    DL_CANDIDATES = [0.025, 0.05, 0.075]
    KL_CANDIDATES = [0.025, 0.05, 0.10]
else:
    TUNING_EPOCHS = min(30, ORIGINAL_EPOCHS)
    CANDIDATE_STABILITY_SEEDS = [1, 2, 3, 4, 5]

    SL_CANDIDATES = [0.0, 0.0025, 0.005, 0.01, 0.02, 0.03, 0.05]
    AL_CANDIDATES = [0.0, 0.01, 0.025, 0.05, 0.075, 0.10]
    DL_CANDIDATES = [0.025, 0.05, 0.075, 0.10]
    KL_CANDIDATES = [0.01, 0.025, 0.05, 0.075, 0.10]

FINAL_EPOCHS = ORIGINAL_EPOCHS

print("MODE:", MODE)
print("Original epochs:", ORIGINAL_EPOCHS)
print("Tuning epochs:", TUNING_EPOCHS)
print("Final epochs:", FINAL_EPOCHS)
print("Candidate stability seeds:", CANDIDATE_STABILITY_SEEDS)

## 8. Add the original loss weights to the candidate lists

This ensures that the original HMRNet-Full setting is always part of the search.

In [ ]:
def get_cfg_value(concept):
    attr = resolved_fields[concept]
    return float(getattr(BASE_CONFIG, attr))

ORIGINAL_WEIGHTS = {
    "stability_weight": get_cfg_value("stability_weight"),
    "alignment_weight": get_cfg_value("alignment_weight"),
    "diversity_weight": get_cfg_value("diversity_weight"),
    "kl_weight": get_cfg_value("kl_weight"),
}

def add_original(candidate_list, value):
    values = list(candidate_list) + [float(value)]
    return sorted(set(round(float(v), 8) for v in values))

SL_CANDIDATES = add_original(
    SL_CANDIDATES, ORIGINAL_WEIGHTS["stability_weight"]
)
AL_CANDIDATES = add_original(
    AL_CANDIDATES, ORIGINAL_WEIGHTS["alignment_weight"]
)
DL_CANDIDATES = add_original(
    DL_CANDIDATES, ORIGINAL_WEIGHTS["diversity_weight"]
)
KL_CANDIDATES = add_original(
    KL_CANDIDATES, ORIGINAL_WEIGHTS["kl_weight"]
)

print("Original loss weights:", ORIGINAL_WEIGHTS)
print("SL candidates:", SL_CANDIDATES)
print("AL candidates:", AL_CANDIDATES)
print("DL candidates:", DL_CANDIDATES)
print("KL candidates:", KL_CANDIDATES)

## 9. Create one fixed tuning/validation split

All screening candidates use **exactly the same document split**, preventing differences in data composition from being confused with hyperparameter effects.

In [ ]:
all_indices = np.arange(n_documents)

tune_idx, validation_idx = train_test_split(
    all_indices,
    test_size=VALIDATION_SIZE,
    random_state=TUNING_SEED,
    shuffle=True,
)

validation_processed_texts = [
    processed_texts[i]
    for i in validation_idx
]

print("Tuning documents:", len(tune_idx))
print("Validation documents:", len(validation_idx))

## 10. Configuration and evaluation helpers

In [ ]:
def clone_config(
    base_config,
    overrides=None,
    seed=None,
    epochs=None,
):
    cfg = copy.deepcopy(base_config)
    overrides = overrides or {}

    for concept, value in overrides.items():
        attr = resolve_config_attr(cfg, concept, required=True)
        setattr(cfg, attr, value)

    if seed is not None:
        setattr(
            cfg,
            resolve_config_attr(cfg, "random_state"),
            int(seed),
        )

    if epochs is not None:
        setattr(
            cfg,
            resolve_config_attr(cfg, "epochs"),
            int(epochs),
        )

    device_attr = resolve_config_attr(
        cfg,
        "device",
        required=False,
    )
    if device_attr is not None:
        setattr(cfg, device_attr, DEVICE)

    patience_attr = resolve_config_attr(
        cfg,
        "patience",
        required=False,
    )
    if patience_attr is not None:
        current_patience = int(getattr(cfg, patience_attr))
        if epochs is not None:
            setattr(
                cfg,
                patience_attr,
                min(current_patience, max(3, int(epochs) // 3)),
            )

    return cfg


def sanitize_name(value):
    value = str(value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_")


def scalar_metrics(metrics):
    output = {}
    for key, value in metrics.items():
        if isinstance(value, (str, int, float, np.integer, np.floating)):
            output[key] = value
    return output


def extract_fusion_summary(model):
    try:
        weights = model.get_fusion_weights()
    except Exception:
        return {}

    if weights is None:
        return {}

    if isinstance(weights, dict):
        result = {}
        for key, value in weights.items():
            arr = np.asarray(value)
            if arr.size:
                result[f"Fusion_{key}"] = float(np.mean(arr))
        return result

    arr = np.asarray(weights)
    if arr.ndim == 1 and arr.size >= 3:
        return {
            "Fusion_Lexical": float(arr[0]),
            "Fusion_Semantic": float(arr[1]),
            "Fusion_Document": float(arr[2]),
        }
    if arr.ndim >= 2 and arr.shape[-1] >= 3:
        avg = np.mean(arr.reshape(-1, arr.shape[-1]), axis=0)
        return {
            "Fusion_Lexical": float(avg[0]),
            "Fusion_Semantic": float(avg[1]),
            "Fusion_Document": float(avg[2]),
        }
    return {}

## 11. Checkpoint-aware training function

Every condition writes both a model checkpoint and a compact `.joblib` result. If Colab disconnects, completed conditions are reused automatically.

In [ ]:
def train_and_evaluate_condition(
    run_name,
    overrides,
    train_indices,
    evaluation_texts,
    seed=TUNING_SEED,
    epochs=TUNING_EPOCHS,
    model_dir=OPT_MODELS_DIR,
    result_dir=OPT_RESULTS_DIR,
    force=False,
):
    safe_name = sanitize_name(run_name)
    model_path = model_dir / f"{safe_name}.pt"
    result_path = result_dir / f"{safe_name}.joblib"

    if result_path.exists() and not force:
        print(f"[CACHE] {run_name}")
        return joblib.load(result_path)

    print("\n" + "=" * 80)
    print("RUN:", run_name)
    print("Overrides:", overrides)
    print("Seed:", seed, "| Epochs:", epochs)
    print("=" * 80)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cfg = clone_config(
        BASE_CONFIG,
        overrides=overrides,
        seed=seed,
        epochs=epochs,
    )

    model = HMRNet(cfg)

    model.fit(
        bow=bow[train_indices],
        semantic_embeddings=semantic_embeddings[train_indices],
        document_embeddings=document_embeddings[train_indices],
        vocabulary=vocabulary,
        verbose=True,
    )

    model.save(model_path)

    topics = model.get_topics(
        top_n=TOP_N_WORDS
    )

    metrics = evaluate_topics(
        model_name=run_name,
        topics=topics,
        processed_texts=evaluation_texts,
        top_n=TOP_N_WORDS,
    )

    row = {
        "Run": run_name,
        "Seed": int(seed),
        "Epochs": int(epochs),
        "Overrides_JSON": json.dumps(overrides, sort_keys=True),
        **overrides,
        **scalar_metrics(metrics),
        **extract_fusion_summary(model),
        "Model_Path": str(model_path),
    }

    payload = {
        "row": row,
        "topics": topics,
        "overrides": overrides,
        "model_path": str(model_path),
    }

    joblib.dump(payload, result_path)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return payload


def row_from_payload(payload):
    return dict(payload["row"])

## 12. Candidate-selection helper

The primary optimization target is \(C_v\), but candidates with very low topic diversity or excessive redundancy are filtered out first.

In [ ]:
def select_best_coherence_row(df):
    if "C_v" not in df.columns:
        raise KeyError(
            "C_v was not returned by evaluate_topics(). "
            f"Available columns: {df.columns.tolist()}"
        )

    eligible = df.copy()

    if "Topic_Diversity" in eligible.columns:
        eligible = eligible[
            eligible["Topic_Diversity"] >= MIN_TOPIC_DIVERSITY
        ]

    if "Topic_Redundancy" in eligible.columns:
        eligible = eligible[
            eligible["Topic_Redundancy"] <= MAX_TOPIC_REDUNDANCY
        ]

    if eligible.empty:
        print(
            "WARNING: No candidate satisfied the diversity/redundancy "
            "constraints. Falling back to all candidates."
        )
        eligible = df.copy()

    sort_cols = ["C_v"]
    ascending = [False]

    if "C_NPMI" in eligible.columns:
        sort_cols.append("C_NPMI")
        ascending.append(False)

    return (
        eligible
        .sort_values(sort_cols, ascending=ascending)
        .iloc[0]
    )


def parse_overrides(row):
    return json.loads(row["Overrides_JSON"])

## 13. Reference run: original HMRNet-Full on the tuning split

This run creates an apples-to-apples reference for the optimization stages.

In [ ]:
import re
reference_payload = train_and_evaluate_condition(
    run_name="stage0_original",
    overrides={},
    train_indices=tune_idx,
    evaluation_texts=validation_processed_texts,
    seed=TUNING_SEED,
    epochs=TUNING_EPOCHS,
    force=FORCE_RETRAIN,
)

reference_row = row_from_payload(reference_payload)

display(
    pd.DataFrame([reference_row])
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

## 14. Stage 1 — Stability-loss weight tuning

The ablation experiment indicated that stability regularization is the first parameter that should be recalibrated. This stage changes **only** the stability-loss weight.

In [ ]:
stage1_payloads = []

for value in SL_CANDIDATES:
    payload = train_and_evaluate_condition(
        run_name=f"stage1_SL_{value:g}",
        overrides={
            "stability_weight": float(value),
        },
        train_indices=tune_idx,
        evaluation_texts=validation_processed_texts,
        seed=TUNING_SEED,
        epochs=TUNING_EPOCHS,
        force=FORCE_RETRAIN,
    )
    stage1_payloads.append(payload)

stage1_df = pd.DataFrame(
    [row_from_payload(p) for p in stage1_payloads]
)

stage1_best = select_best_coherence_row(stage1_df)
best_sl_overrides = parse_overrides(stage1_best)

display(
    stage1_df
    .sort_values("C_v", ascending=False)
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

print("Best Stage-1 overrides:", best_sl_overrides)

## 15. Stage 2 — Alignment-loss weight tuning

The best stability-loss setting is retained while the alignment-loss weight is varied.

In [ ]:
stage2_payloads = []

for value in AL_CANDIDATES:
    overrides = {
        **best_sl_overrides,
        "alignment_weight": float(value),
    }

    payload = train_and_evaluate_condition(
        run_name=(
            f"stage2_SL_{overrides['stability_weight']:g}"
            f"_AL_{value:g}"
        ),
        overrides=overrides,
        train_indices=tune_idx,
        evaluation_texts=validation_processed_texts,
        seed=TUNING_SEED,
        epochs=TUNING_EPOCHS,
        force=FORCE_RETRAIN,
    )
    stage2_payloads.append(payload)

stage2_df = pd.DataFrame(
    [row_from_payload(p) for p in stage2_payloads]
)

stage2_best = select_best_coherence_row(stage2_df)
best_sl_al_overrides = parse_overrides(stage2_best)

display(
    stage2_df
    .sort_values("C_v", ascending=False)
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

print("Best Stage-2 overrides:", best_sl_al_overrides)

## 16. Stage 3 — Diversity-loss weight tuning

The diversity loss is retained rather than removed. This stage searches for a weight that improves coherence while preserving high diversity and low redundancy.

In [ ]:
stage3_payloads = []

for value in DL_CANDIDATES:
    overrides = {
        **best_sl_al_overrides,
        "diversity_weight": float(value),
    }

    payload = train_and_evaluate_condition(
        run_name=(
            f"stage3_SL_{overrides['stability_weight']:g}"
            f"_AL_{overrides['alignment_weight']:g}"
            f"_DL_{value:g}"
        ),
        overrides=overrides,
        train_indices=tune_idx,
        evaluation_texts=validation_processed_texts,
        seed=TUNING_SEED,
        epochs=TUNING_EPOCHS,
        force=FORCE_RETRAIN,
    )
    stage3_payloads.append(payload)

stage3_df = pd.DataFrame(
    [row_from_payload(p) for p in stage3_payloads]
)

stage3_best = select_best_coherence_row(stage3_df)
best_sl_al_dl_overrides = parse_overrides(stage3_best)

display(
    stage3_df
    .sort_values("C_v", ascending=False)
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

print("Best Stage-3 overrides:", best_sl_al_dl_overrides)

## 17. Stage 4 — KL-loss weight tuning

This stage tests whether reducing or increasing variational regularization improves the separability of the learned topics.

In [ ]:
stage4_payloads = []

for value in KL_CANDIDATES:
    overrides = {
        **best_sl_al_dl_overrides,
        "kl_weight": float(value),
    }

    payload = train_and_evaluate_condition(
        run_name=(
            f"stage4_SL_{overrides['stability_weight']:g}"
            f"_AL_{overrides['alignment_weight']:g}"
            f"_DL_{overrides['diversity_weight']:g}"
            f"_KL_{value:g}"
        ),
        overrides=overrides,
        train_indices=tune_idx,
        evaluation_texts=validation_processed_texts,
        seed=TUNING_SEED,
        epochs=TUNING_EPOCHS,
        force=FORCE_RETRAIN,
    )
    stage4_payloads.append(payload)

stage4_df = pd.DataFrame(
    [row_from_payload(p) for p in stage4_payloads]
)

stage4_best = select_best_coherence_row(stage4_df)
coordinate_best_overrides = parse_overrides(stage4_best)

display(
    stage4_df
    .sort_values("C_v", ascending=False)
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

print("Coordinate-search best:", coordinate_best_overrides)

## 18. Optional Stage 5 — Local joint search

Loss terms may interact. The local search takes the two strongest values from each single-coordinate stage and evaluates their combinations. In `quick` mode it is capped to keep runtime manageable.

In [ ]:
def top_values(df, concept, n=2):
    if concept not in df.columns:
        return []
    ranked = (
        df
        .sort_values("C_v", ascending=False)
        [concept]
        .dropna()
        .astype(float)
        .tolist()
    )
    unique = []
    for value in ranked:
        if value not in unique:
            unique.append(value)
        if len(unique) >= n:
            break
    return unique


stage5_payloads = []

if RUN_LOCAL_JOINT_SEARCH:
    local_sl = top_values(stage1_df, "stability_weight", n=2)
    local_al = top_values(stage2_df, "alignment_weight", n=2)
    local_dl = top_values(stage3_df, "diversity_weight", n=2)
    local_kl = top_values(stage4_df, "kl_weight", n=2)

    combinations = list(
        itertools.product(
            local_sl,
            local_al,
            local_dl,
            local_kl,
        )
    )

    if MODE == "quick":
        combinations = combinations[:8]

    print("Local combinations:", len(combinations))

    for sl, al, dl, kl in combinations:
        overrides = {
            "stability_weight": float(sl),
            "alignment_weight": float(al),
            "diversity_weight": float(dl),
            "kl_weight": float(kl),
        }

        payload = train_and_evaluate_condition(
            run_name=(
                f"stage5_SL_{sl:g}_AL_{al:g}"
                f"_DL_{dl:g}_KL_{kl:g}"
            ),
            overrides=overrides,
            train_indices=tune_idx,
            evaluation_texts=validation_processed_texts,
            seed=TUNING_SEED,
            epochs=TUNING_EPOCHS,
            force=FORCE_RETRAIN,
        )
        stage5_payloads.append(payload)

stage5_df = pd.DataFrame(
    [row_from_payload(p) for p in stage5_payloads]
)

if len(stage5_df):
    display(
        stage5_df
        .sort_values("C_v", ascending=False)
        .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
        .round(4)
    )
else:
    print("Local joint search was skipped.")

## 19. Merge all screening runs and save tables

In [ ]:
stage_frames = [
    pd.DataFrame([reference_row]),
    stage1_df,
    stage2_df,
    stage3_df,
    stage4_df,
]

if len(stage5_df):
    stage_frames.append(stage5_df)

all_tuning_df = pd.concat(
    stage_frames,
    ignore_index=True,
    sort=False,
)

all_tuning_df = (
    all_tuning_df
    .drop_duplicates(
        subset=["Overrides_JSON", "Seed", "Epochs"],
        keep="last",
    )
    .sort_values("C_v", ascending=False)
    .reset_index(drop=True)
)

display(
    all_tuning_df
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .head(20)
    .round(4)
)

all_tuning_df.to_csv(
    TABLES_DIR / "experiment_07_all_tuning_runs.csv",
    index=False,
)
all_tuning_df.to_excel(
    TABLES_DIR / "experiment_07_all_tuning_runs.xlsx",
    index=False,
)

stage1_df.to_csv(TABLES_DIR / "experiment_07_stage1_stability_weight.csv", index=False)
stage2_df.to_csv(TABLES_DIR / "experiment_07_stage2_alignment_weight.csv", index=False)
stage3_df.to_csv(TABLES_DIR / "experiment_07_stage3_diversity_weight.csv", index=False)
stage4_df.to_csv(TABLES_DIR / "experiment_07_stage4_kl_weight.csv", index=False)

if len(stage5_df):
    stage5_df.to_csv(
        TABLES_DIR / "experiment_07_stage5_local_joint_search.csv",
        index=False,
    )

print("Screening tables saved.")

## 20. Select top configurations for multi-seed stability validation

The top configurations are deduplicated by their loss-weight settings. Stability validation is performed before selecting the final optimized HMRNet.

In [ ]:
candidate_pool = all_tuning_df.copy()

if "Topic_Diversity" in candidate_pool.columns:
    constrained = candidate_pool[
        candidate_pool["Topic_Diversity"] >= MIN_TOPIC_DIVERSITY
    ].copy()
else:
    constrained = candidate_pool.copy()

if "Topic_Redundancy" in constrained.columns:
    constrained = constrained[
        constrained["Topic_Redundancy"] <= MAX_TOPIC_REDUNDANCY
    ].copy()

if constrained.empty:
    constrained = candidate_pool.copy()

TOP_CONFIGS_FOR_STABILITY = 3 if MODE == "quick" else 5

top_candidates = (
    constrained
    .sort_values("C_v", ascending=False)
    .drop_duplicates("Overrides_JSON")
    .head(TOP_CONFIGS_FOR_STABILITY)
    .reset_index(drop=True)
)

top_candidates.insert(
    0,
    "Candidate_ID",
    [f"C{i+1}" for i in range(len(top_candidates))]
)

display(
    top_candidates[
        [
            col for col in [
                "Candidate_ID",
                "C_v",
                "C_NPMI",
                "Topic_Diversity",
                "Topic_Redundancy",
                "Overrides_JSON",
            ]
            if col in top_candidates.columns
        ]
    ].round(4)
)

## 21. Multi-seed stability validation of the shortlisted candidates

For each candidate, HMRNet is retrained using several independent random seeds. Topics are matched across runs with the same stability procedure used in Experiment 03.

In [ ]:
candidate_stability_rows = []
candidate_pairwise_frames = []

full_indices = np.arange(n_documents)

for _, candidate in top_candidates.iterrows():
    candidate_id = candidate["Candidate_ID"]
    overrides = parse_overrides(candidate)

    beta_by_seed = {}

    print("\n" + "#" * 80)
    print("STABILITY CANDIDATE:", candidate_id)
    print("Overrides:", overrides)
    print("#" * 80)

    for seed in CANDIDATE_STABILITY_SEEDS:
        run_name = f"stability_{candidate_id}_seed_{seed}"

        payload = train_and_evaluate_condition(
            run_name=run_name,
            overrides=overrides,
            train_indices=full_indices,
            evaluation_texts=processed_texts,
            seed=seed,
            epochs=TUNING_EPOCHS,
            model_dir=OPT_STABILITY_DIR,
            result_dir=OPT_STABILITY_DIR,
            force=FORCE_RETRAIN,
        )

        model_path = Path(payload["model_path"])
        model = HMRNet.load(
            model_path,
            device=DEVICE,
        )

        beta_by_seed[seed] = (
            model
            .get_topic_word_distribution()
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pairwise, summary = evaluate_stability_runs(
        beta_by_run=beta_by_seed,
        metric="js",
        confidence=0.95,
    )

    pairwise["Candidate_ID"] = candidate_id
    pairwise["Overrides_JSON"] = candidate["Overrides_JSON"]
    candidate_pairwise_frames.append(pairwise)

    candidate_stability_rows.append(
        {
            "Candidate_ID": candidate_id,
            "Validation_C_v": float(candidate["C_v"]),
            "Validation_C_NPMI": (
                float(candidate["C_NPMI"])
                if "C_NPMI" in candidate
                and pd.notna(candidate["C_NPMI"])
                else np.nan
            ),
            "Validation_Topic_Diversity": (
                float(candidate["Topic_Diversity"])
                if "Topic_Diversity" in candidate
                and pd.notna(candidate["Topic_Diversity"])
                else np.nan
            ),
            "Validation_Topic_Redundancy": (
                float(candidate["Topic_Redundancy"])
                if "Topic_Redundancy" in candidate
                and pd.notna(candidate["Topic_Redundancy"])
                else np.nan
            ),
            "Overrides_JSON": candidate["Overrides_JSON"],
            **summary,
        }
    )

candidate_stability_df = pd.DataFrame(
    candidate_stability_rows
).sort_values(
    ["Validation_C_v", "Mean_TS"],
    ascending=[False, False],
).reset_index(drop=True)

candidate_pairwise_df = pd.concat(
    candidate_pairwise_frames,
    ignore_index=True,
)

display(candidate_stability_df.round(4))

candidate_stability_df.to_csv(
    TABLES_DIR / "experiment_07_candidate_stability.csv",
    index=False,
)
candidate_stability_df.to_excel(
    TABLES_DIR / "experiment_07_candidate_stability.xlsx",
    index=False,
)
candidate_pairwise_df.to_csv(
    TABLES_DIR / "experiment_07_candidate_stability_pairwise.csv",
    index=False,
)

## 22. Select the optimized configuration

Primary rule:

1. satisfy the minimum stability threshold;
2. satisfy diversity/redundancy constraints from screening;
3. maximize validation \(C_v\).

If no candidate reaches the stability threshold, the notebook falls back to a coherence–stability composite score and reports the fallback explicitly.

In [ ]:
eligible_final = candidate_stability_df[
    candidate_stability_df["Mean_TS"] >= MIN_TOPIC_STABILITY
].copy()

selection_rule = (
    f"Highest validation C_v among candidates with "
    f"Mean_TS >= {MIN_TOPIC_STABILITY}"
)

if eligible_final.empty:
    print(
        "WARNING: No candidate reached the required stability threshold. "
        "Using a fallback coherence-stability composite score."
    )

    temp = candidate_stability_df.copy()

    def minmax(series):
        series = series.astype(float)
        if np.isclose(series.max(), series.min()):
            return pd.Series(
                np.ones(len(series)),
                index=series.index,
            )
        return (series - series.min()) / (series.max() - series.min())

    temp["Composite"] = (
        0.65 * minmax(temp["Validation_C_v"])
        + 0.35 * minmax(temp["Mean_TS"])
    )

    best_candidate = temp.sort_values(
        "Composite",
        ascending=False,
    ).iloc[0]

    selection_rule = "Fallback: 0.65 normalized C_v + 0.35 normalized Mean_TS"
else:
    best_candidate = eligible_final.sort_values(
        ["Validation_C_v", "Mean_TS"],
        ascending=[False, False],
    ).iloc[0]

BEST_OVERRIDES = json.loads(
    best_candidate["Overrides_JSON"]
)

print("Selected candidate:", best_candidate["Candidate_ID"])
print("Selection rule:", selection_rule)
print("Validation C_v:", best_candidate["Validation_C_v"])
print("Mean topic stability:", best_candidate["Mean_TS"])
print("Best overrides:")
print(json.dumps(BEST_OVERRIDES, indent=2))

## 23. Train the final optimized HMRNet on the complete corpus

The selected hyperparameters are now retrained using all documents and the original paper-quality epoch budget.

In [ ]:
final_payload = train_and_evaluate_condition(
    run_name="HMRNet_Optimized_Final",
    overrides=BEST_OVERRIDES,
    train_indices=full_indices,
    evaluation_texts=processed_texts,
    seed=TUNING_SEED,
    epochs=FINAL_EPOCHS,
    model_dir=OPT_MODELS_DIR,
    result_dir=OPT_RESULTS_DIR,
    force=FORCE_RETRAIN,
)

final_row = row_from_payload(final_payload)
final_source_path = Path(final_payload["model_path"])

FINAL_MODEL_PATH = CHECKPOINTS_DIR / "HMRNet-Optimized.pt"

# Copy the final checkpoint to the standard checkpoint folder.
import shutil
shutil.copy2(
    final_source_path,
    FINAL_MODEL_PATH,
)

final_metrics_df = pd.DataFrame(
    [final_row]
)

display(
    final_metrics_df
    .drop(columns=["Overrides_JSON", "Model_Path"], errors="ignore")
    .round(4)
)

final_metrics_df.to_csv(
    TABLES_DIR / "experiment_07_final_metrics.csv",
    index=False,
)
final_metrics_df.to_excel(
    TABLES_DIR / "experiment_07_final_metrics.xlsx",
    index=False,
)

best_config_payload = {
    "selection_rule": selection_rule,
    "candidate_id": str(best_candidate["Candidate_ID"]),
    "validation_C_v": float(best_candidate["Validation_C_v"]),
    "candidate_mean_topic_stability": float(best_candidate["Mean_TS"]),
    "overrides": BEST_OVERRIDES,
    "final_epochs": int(FINAL_EPOCHS),
    "final_seed": int(TUNING_SEED),
    "final_model_path": str(FINAL_MODEL_PATH),
}

with open(
    TABLES_DIR / "experiment_07_best_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_config_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Final optimized checkpoint:", FINAL_MODEL_PATH)

## 24. Compare against previous Experiment 01/02 results

This table is intended for interpretation only. It preserves the original reported values rather than replacing earlier experiment files.

In [ ]:
comparison_rows = []

baseline_path = TABLES_DIR / "experiment_01_baseline_comparison.csv"
ablation_path = TABLES_DIR / "experiment_02_ablation.csv"

if baseline_path.exists():
    baseline_df = pd.read_csv(baseline_path)
    full_match = baseline_df[
        baseline_df["Model"].astype(str).str.contains(
            "HMRNet-Full|HMRNet_Full",
            regex=True,
        )
    ]
    if len(full_match):
        row = full_match.iloc[0]
        comparison_rows.append(
            {
                "Configuration": "Original HMRNet-Full (Experiment 01)",
                "C_v": row.get("C_v", np.nan),
                "C_NPMI": row.get("C_NPMI", np.nan),
                "Topic_Diversity": row.get("Topic_Diversity", np.nan),
                "Topic_Redundancy": row.get("Topic_Redundancy", np.nan),
            }
        )

if ablation_path.exists():
    ablation_df = pd.read_csv(ablation_path)
    sl_match = ablation_df[
        ablation_df["Model"].astype(str).str.contains(
            "HMRNet_-SL|HMRNet-.*SL",
            regex=True,
        )
    ]
    if len(sl_match):
        row = sl_match.iloc[0]
        comparison_rows.append(
            {
                "Configuration": "HMRNet without Stability Loss (Experiment 02)",
                "C_v": row.get("C_v", np.nan),
                "C_NPMI": row.get("C_NPMI", np.nan),
                "Topic_Diversity": row.get("Topic_Diversity", np.nan),
                "Topic_Redundancy": row.get("Topic_Redundancy", np.nan),
            }
        )

comparison_rows.append(
    {
        "Configuration": "HMRNet-Optimized (Experiment 07)",
        "C_v": final_row.get("C_v", np.nan),
        "C_NPMI": final_row.get("C_NPMI", np.nan),
        "Topic_Diversity": final_row.get("Topic_Diversity", np.nan),
        "Topic_Redundancy": final_row.get("Topic_Redundancy", np.nan),
    }
)

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.round(4))

comparison_df.to_csv(
    TABLES_DIR / "experiment_07_comparison.csv",
    index=False,
)
comparison_df.to_excel(
    TABLES_DIR / "experiment_07_comparison.xlsx",
    index=False,
)

## 25. Optimization plots

In [ ]:
# Plot 1: stability-loss screening
if len(stage1_df):
    plot_df = stage1_df.sort_values("stability_weight")

    plt.figure(figsize=(7, 4.5))
    plt.plot(
        plot_df["stability_weight"],
        plot_df["C_v"],
        marker="o",
    )
    plt.xlabel("Stability-loss weight")
    plt.ylabel("Validation C_v")
    plt.title("HMRNet Coherence vs Stability-Loss Weight")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "experiment_07_stability_weight_cv.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

# Plot 2: candidate coherence vs stability
plt.figure(figsize=(7, 5))

plt.scatter(
    candidate_stability_df["Validation_C_v"],
    candidate_stability_df["Mean_TS"],
    s=70,
)

for _, row in candidate_stability_df.iterrows():
    plt.annotate(
        row["Candidate_ID"],
        (
            row["Validation_C_v"],
            row["Mean_TS"],
        ),
        xytext=(5, 5),
        textcoords="offset points",
    )

plt.axhline(
    MIN_TOPIC_STABILITY,
    linestyle="--",
    linewidth=1,
)

plt.xlabel("Validation C_v")
plt.ylabel("Mean Topic Stability")
plt.title("Coherence–Stability Trade-off")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_07_coherence_stability_tradeoff.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 26. Optional: final 10-seed stability evaluation

For the manuscript-quality experiment, set `MODE="final"`. The notebook then evaluates the selected optimized configuration using ten random seeds, matching the design of Experiment 03.

This is computationally expensive but strongly recommended before replacing the original HMRNet-Full result in the paper.

In [ ]:
final_stability_summary_df = pd.DataFrame()
final_stability_pairwise_df = pd.DataFrame()

if RUN_FINAL_10_SEED_STABILITY:
    FINAL_STABILITY_SEEDS = list(range(1, 11))
    beta_by_seed = {}

    final_stability_model_dir = (
        OPT_STABILITY_DIR / "final_10_seed"
    )
    final_stability_result_dir = (
        OPT_STABILITY_DIR / "final_10_seed_results"
    )

    final_stability_model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )
    final_stability_result_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for seed in FINAL_STABILITY_SEEDS:
        payload = train_and_evaluate_condition(
            run_name=f"optimized_final_seed_{seed}",
            overrides=BEST_OVERRIDES,
            train_indices=full_indices,
            evaluation_texts=processed_texts,
            seed=seed,
            epochs=FINAL_EPOCHS,
            model_dir=final_stability_model_dir,
            result_dir=final_stability_result_dir,
            force=FORCE_RETRAIN,
        )

        model = HMRNet.load(
            Path(payload["model_path"]),
            device=DEVICE,
        )

        beta_by_seed[seed] = (
            model
            .get_topic_word_distribution()
        )

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pairwise, summary = evaluate_stability_runs(
        beta_by_run=beta_by_seed,
        metric="js",
        confidence=0.95,
    )

    final_stability_pairwise_df = pairwise.copy()
    final_stability_summary_df = pd.DataFrame(
        [
            {
                "Model": "HMRNet-Optimized",
                **summary,
            }
        ]
    )

    display(final_stability_summary_df.round(4))

    final_stability_pairwise_df.to_csv(
        TABLES_DIR / "experiment_07_final_stability_pairwise.csv",
        index=False,
    )
    final_stability_summary_df.to_csv(
        TABLES_DIR / "experiment_07_final_stability_summary.csv",
        index=False,
    )
    final_stability_summary_df.to_excel(
        TABLES_DIR / "experiment_07_final_stability_summary.xlsx",
        index=False,
    )
else:
    print(
        "Final 10-seed stability is disabled. "
        "Set MODE='final' and rerun this notebook for manuscript-quality validation."
    )

## 27. Final verification

In [ ]:
required_outputs = [
    TABLES_DIR / "experiment_07_all_tuning_runs.csv",
    TABLES_DIR / "experiment_07_candidate_stability.csv",
    TABLES_DIR / "experiment_07_final_metrics.csv",
    TABLES_DIR / "experiment_07_best_config.json",
    TABLES_DIR / "experiment_07_comparison.csv",
    FIGURES_DIR / "experiment_07_stability_weight_cv.png",
    FIGURES_DIR / "experiment_07_coherence_stability_tradeoff.png",
    CHECKPOINTS_DIR / "HMRNet-Optimized.pt",
]

if RUN_FINAL_10_SEED_STABILITY:
    required_outputs.extend(
        [
            TABLES_DIR / "experiment_07_final_stability_pairwise.csv",
            TABLES_DIR / "experiment_07_final_stability_summary.csv",
        ]
    )

print("Experiment 07 outputs:")
for path in required_outputs:
    print(f"{'✓' if path.exists() else '✗'} {path}")

print("\nBest configuration:")
print(json.dumps(BEST_OVERRIDES, indent=2))

print("\nFinal metrics:")
for key in [
    "C_v",
    "C_NPMI",
    "UMass",
    "Topic_Diversity",
    "Topic_Redundancy",
    "Topic_Uniqueness",
]:
    if key in final_row:
        print(f"  {key}: {final_row[key]}")

## Interpretation guide for the manuscript

Use Experiment 07 only after the final-mode run has completed.

A defensible interpretation is:

- **coherence improves** if optimized \(C_v\) exceeds the original HMRNet-Full value under a comparable evaluation setting;
- **diversity is preserved** if Topic Diversity remains high and Topic Redundancy remains low;
- **stability is preserved** if the optimized multi-seed Topic Stability remains above the pre-specified threshold;
- avoid selecting a model solely because it produces the maximum \(C_v\);
- if the optimized model is used in the final manuscript, rerun downstream utility and automatic interpretability with the optimized checkpoint so that all reported HMRNet results refer to the same final configuration.

### Important methodological note

Hyperparameter tuning can make Experiment 07 a **model-selection stage** rather than an additional performance benchmark. For a Q1 manuscript, describe the search space, fixed data split, selection constraints, random seeds, and final retraining procedure explicitly.